# Multi-Surface Configuration: Wing and Stabilizers

In this notebook, I analyze a configuration consisting of a main wing, horizontal stabilizer (HS), and vertical stabilizer (VS) with the `VLM` class. I inspect each surface's contribution and vary the HS incidence to observe the response of the combined configuration.

I use the coupled wing-tail study in Section 6.8 of my thesis as a reference and add a VS to exercise the multi-surface API. Therefore, this is an illustrative case and does not numerically reproduce the thesis table.

The surfaces are coupled by solving one influence matrix for all panels. The model does not include a fuselage or a free wake that evolves downstream.

In [63]:
import sys
from pathlib import Path


def _find_repo_root(start: Path) -> Path:
    current = start
    while current != current.parent:
        if (current / 'src' / 'vlm').is_dir():
            return current
        current = current.parent
    raise RuntimeError('Could not locate the VLMPy repository root (expected src/vlm).')


REPO_ROOT = _find_repo_root(Path.cwd())
SRC_VLM = REPO_ROOT / 'src' / 'vlm'
if str(SRC_VLM) not in sys.path:
    sys.path.insert(0, str(SRC_VLM))

print(f'VLM source path added to sys.path: {SRC_VLM}')

VLM source path added to sys.path: c:\Users\javil\OneDrive - Universidad Alfonso X el Sabio\Documents\GitHub\VLMPy\src\vlm


In [64]:
import contextlib
import io

import numpy as np
import plotly.graph_objects as go

from lib.vlm import VLM, plot_distribution_all

## 1. Configuration Definition

Each surface is defined by one section. The `x_translate` and `z_translate` fields position the stabilizers relative to the wing origin; `alpha` specifies their incidence in radians and does not replace the flow angle of attack.

In this model, the VS is oriented with `dihedral = 90°` and treated as a non-symmetric surface. This convention describes the panel geometry in this implementation; it does not, by itself, add a fuselage, movable rudder, or realistic lateral-flow model.

In [65]:
plane = {
    'wing_sections': [
        {
            'chord_root': 1.0,
            'chord_tip': 0.8,
            'span_fraction': 10.0,
            'sweep': np.radians(30.0),
            'dihedral': np.radians(5.0),
            'NACA_root': '4412',
            'NACA_tip': '4412',
        }
    ],
    'horizontal_stabilizer': {
        'x_translate': 6.0,
        'z_translate': 0.0,
        'NACA_root': '1012',
        'NACA_tip': '1012',
        'chord_root': 0.5,
        'chord_tip': 0.5,
        'span_fraction': 4.0,
        'sweep': np.radians(20.0),
        'dihedral': np.radians(6.0),
        'alpha': 0.0,
    },
    'vertical_stabilizer': {
        'x_translate': 6.0,
        'z_translate': 0.0,
        'NACA_root': '0012',
        'NACA_tip': '0012',
        'chord_root': 0.6,
        'chord_tip': 0.4,
        'span_fraction': 1.5,
        'sweep': np.radians(15.0),
        'dihedral': np.radians(90.0),
        'alpha': 0.0,
    },
}

vlm = VLM(plane=plane, u=50.0, rho=1.225, alpha=np.radians(5.0), beta=0.0, n=40, m=8)

## 2. Solution and Per-Surface Contributions

I check that the solver returns finite coefficients and that the total panel count matches the sum of the wing, HS, and VS discretizations. I then compare forces by surface; their magnitudes depend on each component's area and incidence.

In [66]:
_out = io.StringIO()
with contextlib.redirect_stdout(_out):
    vlm.run_vlm()

print(f'Total  : CL={vlm.CL:.4f}  CD={vlm.CD:.5f}')
print(f'Wing   : lift={np.sum(vlm.lift_wing):.2f} N  (n={vlm.n}, m={vlm.m})')
print(f'HS     : lift={np.sum(vlm.lift_hs):.2f} N  (n={vlm.n_hs}, m={vlm.m_hs})')
print(f'VS     : lift={np.sum(vlm.lift_vs):.2f} N  (n={vlm.n_vs}, m={vlm.m_vs})')

assert np.isfinite(vlm.CL) and np.isfinite(vlm.CD)
expected_panels = vlm.n * vlm.m + vlm.n_hs * vlm.m_hs + vlm.n_vs * vlm.m_vs
assert len(vlm.panel_data) == expected_panels

Total  : CL=0.5878  CD=0.00763
Wing   : lift=10972.77 N  (n=40, m=8)
HS     : lift=-396.37 N  (n=16, m=4)
VS     : lift=-0.00 N  (n=6, m=5)


## 3. Local $C_L$ Distribution

I plot the local distributions for all surfaces in the same figure. The comparison helps identify how spanwise loading differs; it should not be read as a comparison of 2D airfoils or surface pressure.

In [67]:
fig = go.Figure()
fig = plot_distribution_all(fig, vlm, quantity='CL')
fig.show()

## 4. Sensitivity to Horizontal-Stabilizer Incidence

I sweep the HS incidence while keeping the wing geometry, VS, and flight conditions fixed. I record HS and wing lift separately: I expect the largest response on the surface whose incidence changes, although coupling may also alter the wing load.

This is a qualitative comparison of the solver over the selected range; it is not validation against experimental data or a longitudinal-stability calculation.

In [73]:
hs_incidences_deg = [-4.0, -2.0, 0.0, 2.0, 4.0]
hs_lift_totals = []
wing_lift_totals = []

for incidence_deg in hs_incidences_deg:
    plane_case = {
        'wing_sections': [dict(plane['wing_sections'][0])],
        'horizontal_stabilizer': dict(plane['horizontal_stabilizer']),
        'vertical_stabilizer': dict(plane['vertical_stabilizer']),
    }
    plane_case['horizontal_stabilizer']['alpha'] = np.radians(incidence_deg)

    vlm_case = VLM(plane=plane_case, u=50.0, rho=1.225, alpha=np.radians(5.0), beta=0.0, n=40, m=8)
    _out = io.StringIO()
    with contextlib.redirect_stdout(_out):
        vlm_case.run_vlm()
    hs_lift_totals.append(float(np.sum(vlm_case.lift_hs)))
    wing_lift_totals.append(float(np.sum(vlm_case.lift_wing)))

print('HS incidence [deg] -> HS lift [N] | wing lift [N]:')
for incidence_deg, hs_lift, wing_lift in zip(hs_incidences_deg, hs_lift_totals, wing_lift_totals):
    print(f'  {incidence_deg:+.1f}  ->  {hs_lift:+8.2f}  |  {wing_lift:8.2f}')

assert hs_lift_totals[-1] > hs_lift_totals[0], 'HS lift should increase monotonically with incidence over this range.'
wing_lift_span = max(wing_lift_totals) - min(wing_lift_totals)
hs_lift_span = max(hs_lift_totals) - min(hs_lift_totals)
print(f'Wing lift variation over the sweep: {wing_lift_span:.2f} N (HS lift variation: {hs_lift_span:.2f} N)')

HS incidence [deg] -> HS lift [N] | wing lift [N]:
  -4.0  ->  -1422.95  |  10937.39
  -2.0  ->   -909.32  |  10955.10
  +0.0  ->   -396.37  |  10972.77
  +2.0  ->   +117.07  |  10990.44
  +4.0  ->   +632.55  |  11008.15
Wing lift variation over the sweep: 70.76 N (HS lift variation: 2055.50 N)


### 4.1. Per-Surface Response to Incidence

I present wing and stabilizer loads separately because their magnitudes differ substantially. Each figure has its own vertical scale; comparing the qualitative slopes is more informative than plotting both forces on a single axis.

In [74]:
wing_incidence_figure = go.Figure()
wing_incidence_figure.add_trace(go.Scatter(
    x=hs_incidences_deg, y=wing_lift_totals,
    mode='lines+markers', name='Main wing',
))
wing_incidence_figure.update_layout(
    title='Wing lift versus HS incidence',
    xaxis_title='HS incidence [deg]',
    yaxis_title='Wing lift [N]',
    template='plotly_white',
)
wing_incidence_figure.show()

hs_incidence_figure = go.Figure()
hs_incidence_figure.add_trace(go.Scatter(
    x=hs_incidences_deg, y=hs_lift_totals,
    mode='lines+markers', name='Horizontal stabilizer',
))
hs_incidence_figure.update_layout(
    title='HS lift versus its incidence',
    xaxis_title='HS incidence [deg]',
    yaxis_title='HS lift [N]',
    template='plotly_white',
)
hs_incidence_figure.show()
assert np.isfinite(wing_lift_totals).all()
assert np.isfinite(hs_lift_totals).all()

## 5. Scope, Results, and Limitations

For the baseline configuration, with $\alpha=5^\circ$ and zero HS incidence, the solver gives $C_L=0.5878$ and $C_D=0.00763$. The lift contributions are 10,972.77 N from the wing, -396.37 N from the HS, and approximately 0 N from the VS. The stabilizer's negative contribution is consistent with operation in the wing's induced field; this model does not solve a free wake to verify that mechanism independently.

Across the incidence sweep, HS lift increases from -1,422.95 N to +632.55 N between -4° and +4°. The change in spanwise wing load is much smaller: 70.76 N versus 2,055.50 N for the HS. This supports the qualitative trend that incidence primarily changes the stabilizer load, but it does not mean the wing is unaffected.

- The fuselage is not modeled; forces correspond only to the defined lifting surfaces.
- Inter-surface influence is represented by the solver's shared matrix, not by a deforming wake or free-vortex downstream model.
- The force change with incidence is not a validated stability derivative and does not, by itself, predict trim.
- The model does not include viscosity, separation, stall, or compressibility. The calculated $C_D$ is induced drag, not total aircraft drag.

## 6. Sideslip in a Multi-Surface Configuration

I extend the HS/VS study to observe each component's load as $\beta$ changes. This is not the configuration from thesis case 6.5.1. Also, `case.CL` sums scalar magnitudes from all three surfaces and is not equivalent to an aircraft vertical-lift coefficient when the VS produces lateral force. I therefore separate wing and HS lift from the VS normal force instead of interpreting the aggregate as conventional `CL`.

In [70]:
import importlib
import lib.vlm as vlm_module
importlib.reload(vlm_module)
VLM = vlm_module.VLM

## 7. Main-Wing Sideslip: Thesis Case 6.5.1

In Section 6.5.1, I analyzed a NACA 4412 wing with a 10 m span, 1 m chord, 30° sweep, 5° dihedral, and $\alpha=0^\circ$, at $U_\infty=50\,\mathrm{m/s}$ and $\rho=1.225\,\mathrm{kg/m^3}$. Here I model only that wing, without HS or VS; I use a demonstrative `(30, 6)` mesh, so this is not a numerical reproduction of the thesis mesh `(100, 10)`.

The thesis convention defines $V_{\infty,y}=-U_\infty\cos\alpha\sin\beta$. Thus, for $\beta>0$, the flow approaches from the `+y` side. Because this case contains only the wing, I can check that global $C_L$ and $C_{D_i}$ are even functions of beta and that the local distribution mirrors across the two semispans.

In [75]:
beta_multisurface_deg = [-12.0, -6.0, 0.0, 6.0, 12.0]
wing_lift_multisurface = []
hs_lift_multisurface = []
vs_lift_scalar_multisurface = []
wing_cl_multisurface = []
cd_multisurface = []
wing_cl_local_multisurface = []
span_positions_multisurface = None

for beta_deg in beta_multisurface_deg:
    case = VLM(
        plane=plane, u=50.0, rho=1.225,
        alpha=0.0, beta=np.radians(beta_deg),
        n=30, m=6,
    )
    captured = io.StringIO()
    with contextlib.redirect_stdout(captured):
        case.run_vlm()
    assert (case.n, case.m) == (30, 6)
    wing_panels = case.panel_data[:case.n * case.m]
    span_positions = np.array([
        np.mean([
            panel[3][0]
            for panel in wing_panels[strip * case.m:(strip + 1) * case.m]
        ])
        for strip in range(case.n)
    ])
    if span_positions_multisurface is None:
        span_positions_multisurface = span_positions
    else:
        np.testing.assert_allclose(span_positions, span_positions_multisurface, atol=1e-12)

    dynamic_pressure_area = 0.5 * case.rho * case.u**2 * case.wing_area
    wing_lift_multisurface.append(float(np.sum(case.lift_wing)))
    hs_lift_multisurface.append(float(np.sum(case.lift_hs)))
    vs_lift_scalar_multisurface.append(float(np.sum(case.lift_vs)))
    wing_cl_multisurface.append(wing_lift_multisurface[-1] / dynamic_pressure_area)
    cd_multisurface.append(float(case.CD))
    wing_cl_local_multisurface.append(np.asarray(case.CL_locals_wing, dtype=float))
    print(
        'beta={:+.0f}°: wing CL={:.6f}; wing/HS lift={:.2f}/{:.2f} N; VS scalar output={:.2f} N; induced CD={:.6f}'.format(
            beta_deg, wing_cl_multisurface[-1],
            wing_lift_multisurface[-1], hs_lift_multisurface[-1],
            vs_lift_scalar_multisurface[-1], cd_multisurface[-1]
        )
    )

for positive_index, negative_index in ((0, 4), (1, 3)):
    np.testing.assert_allclose(
        wing_cl_local_multisurface[positive_index],
        wing_cl_local_multisurface[negative_index][::-1],
        rtol=1e-8, atol=1e-10,
    )
    np.testing.assert_allclose(
        wing_cl_multisurface[positive_index], wing_cl_multisurface[negative_index],
        rtol=1e-8, atol=1e-10,
    )
    np.testing.assert_allclose(
        hs_lift_multisurface[positive_index], hs_lift_multisurface[negative_index],
        rtol=1e-8, atol=1e-8,
    )
    np.testing.assert_allclose(
        vs_lift_scalar_multisurface[positive_index],
        -vs_lift_scalar_multisurface[negative_index],
        rtol=1e-8, atol=1e-8,
    )
    np.testing.assert_allclose(
        cd_multisurface[positive_index], cd_multisurface[negative_index],
        rtol=1e-8, atol=1e-10,
    )

center_index = beta_multisurface_deg.index(0.0)
np.testing.assert_allclose(
    wing_cl_local_multisurface[center_index],
    wing_cl_local_multisurface[center_index][::-1],
    rtol=1e-8, atol=1e-10,
 )
positive_beta_index = beta_multisurface_deg.index(6.0)
positive_y_mean = np.mean(
    wing_cl_local_multisurface[positive_beta_index][span_positions_multisurface > 0.0]
 )
negative_y_mean = np.mean(
    wing_cl_local_multisurface[positive_beta_index][span_positions_multisurface < 0.0]
 )
assert positive_y_mean > negative_y_mean

multisurface_local_figure = go.Figure()
for beta_deg, local_cl in zip(beta_multisurface_deg, wing_cl_local_multisurface):
    multisurface_local_figure.add_trace(go.Scatter(
        x=span_positions_multisurface, y=local_cl,
        mode='lines+markers', name='beta={:+.0f}°'.format(beta_deg),
    ))
multisurface_local_figure.update_layout(
    title='Local wing load versus sideslip',
    xaxis_title='Spanwise coordinate y [m]',
    yaxis_title='Wing local CL [-]', template='plotly_white',
 )
multisurface_local_figure.show()

component_figure = go.Figure()
component_figure.add_trace(go.Scatter(
    x=beta_multisurface_deg, y=hs_lift_multisurface,
    mode='lines+markers', name='Reported lift_hs [N]',
))
component_figure.add_trace(go.Scatter(
    x=beta_multisurface_deg, y=vs_lift_scalar_multisurface,
    mode='lines+markers', name='Reported VS scalar output [N]',
))
component_figure.update_layout(
    title='Reported scalar contributions from HS and VS',
    xaxis_title='Beta [deg]',
    yaxis_title='Solver output [N]', template='plotly_white',
 )
component_figure.show()

wing_cl_figure = go.Figure()
wing_cl_figure.add_trace(go.Scatter(
    x=beta_multisurface_deg, y=wing_cl_multisurface,
    mode='lines+markers', name='Wing CL',
))
wing_cl_figure.update_layout(
    title='Wing CL versus sideslip',
    xaxis_title='Beta [deg]', yaxis_title='Wing CL [-]',
    template='plotly_white',
 )
wing_cl_figure.show()

assert np.isfinite(wing_cl_multisurface).all()
assert np.isfinite(vs_lift_scalar_multisurface).all()

beta=-12°: wing CL=0.244504; wing/HS lift=4399.15/-159.23 N; VS scalar output=-952.79 N; induced CD=0.004364
beta=-6°: wing CL=0.248597; wing/HS lift=4472.80/-161.89 N; VS scalar output=-479.02 N; induced CD=0.002053
beta=+0°: wing CL=0.249966; wing/HS lift=4497.43/-162.79 N; VS scalar output=0.00 N; induced CD=0.001271
beta=+6°: wing CL=0.248597; wing/HS lift=4472.80/-161.89 N; VS scalar output=479.02 N; induced CD=0.002053
beta=+12°: wing CL=0.244504; wing/HS lift=4399.15/-159.23 N; VS scalar output=952.79 N; induced CD=0.004364


In [76]:
plane_beta = {
    'wing_sections': [{
        'chord_root': 1.0, 'chord_tip': 1.0,
        'span_fraction': 10.0,
        'sweep': np.radians(30.0),
        'dihedral': np.radians(5.0),
        'NACA_root': '4412', 'NACA_tip': '4412',
    }]
}
beta_wing_deg = [-12.0, -6.0, 0.0, 6.0, 12.0]
cl_global_wing_beta = []
cd_wing_beta = []
cl_local_wing_beta = []
span_positions_wing_beta = None

for beta_deg in beta_wing_deg:
    case = VLM(
        plane=plane_beta, u=50.0, rho=1.225,
        alpha=0.0, beta=np.radians(beta_deg),
        n=30, m=6,
    )
    captured = io.StringIO()
    with contextlib.redirect_stdout(captured):
        case.run_vlm()
    assert (case.n, case.m) == (30, 6)
    span_positions = np.array([
        np.mean([
            panel[3][0]
            for panel in case.panel_data[strip * case.m:(strip + 1) * case.m]
        ])
        for strip in range(case.n)
    ])
    if span_positions_wing_beta is None:
        span_positions_wing_beta = span_positions
    else:
        np.testing.assert_allclose(span_positions, span_positions_wing_beta, atol=1e-12)
    cl_global_wing_beta.append(float(case.CL))
    cd_wing_beta.append(float(case.CD))
    cl_local_wing_beta.append(np.asarray(case.CL_locals_wing, dtype=float))
    print(
        'beta={:+.0f}°: CL={:.6f}, induced CD={:.6f}, panels={}'.format(
            beta_deg, case.CL, case.CD, len(case.panel_data)
        )
    )

for positive_index, negative_index in ((0, 4), (1, 3)):
    np.testing.assert_allclose(
        cl_local_wing_beta[positive_index],
        cl_local_wing_beta[negative_index][::-1],
        rtol=1e-8, atol=1e-10,
    )
    np.testing.assert_allclose(
        cl_global_wing_beta[positive_index], cl_global_wing_beta[negative_index],
        rtol=1e-8, atol=1e-10,
    )
    np.testing.assert_allclose(
        cd_wing_beta[positive_index], cd_wing_beta[negative_index],
        rtol=1e-8, atol=1e-10,
    )

center_index = beta_wing_deg.index(0.0)
np.testing.assert_allclose(
    cl_local_wing_beta[center_index],
    cl_local_wing_beta[center_index][::-1],
    rtol=1e-8, atol=1e-10,
 )
positive_beta_index = beta_wing_deg.index(6.0)
positive_y_mean = np.mean(
    cl_local_wing_beta[positive_beta_index][span_positions_wing_beta > 0.0]
 )
negative_y_mean = np.mean(
    cl_local_wing_beta[positive_beta_index][span_positions_wing_beta < 0.0]
 )
assert positive_y_mean > negative_y_mean

wing_local_figure = go.Figure()
for beta_deg, local_cl in zip(beta_wing_deg, cl_local_wing_beta):
    wing_local_figure.add_trace(go.Scatter(
        x=span_positions_wing_beta, y=local_cl,
        mode='lines+markers', name='beta={:+.0f}°'.format(beta_deg),
    ))
wing_local_figure.update_layout(
    title='Local wing CL versus sideslip',
    xaxis_title='Spanwise coordinate y [m]',
    yaxis_title='Local CL [-]', template='plotly_white',
 )
wing_local_figure.show()

wing_global_cl_figure = go.Figure()
wing_global_cl_figure.add_trace(go.Scatter(
    x=beta_wing_deg, y=cl_global_wing_beta,
    mode='lines+markers', name='Global CL',
))
wing_global_cl_figure.update_layout(
    title='Global wing CL versus beta',
    xaxis_title='Beta [deg]', yaxis_title='CL [-]',
    template='plotly_white',
 )
wing_global_cl_figure.show()

wing_global_cd_figure = go.Figure()
wing_global_cd_figure.add_trace(go.Scatter(
    x=beta_wing_deg, y=cd_wing_beta,
    mode='lines+markers', name='Induced CD',
))
wing_global_cd_figure.update_layout(
    title='Wing induced CD versus beta',
    xaxis_title='Beta [deg]', yaxis_title='Induced CD [-]',
    template='plotly_white',
 )
wing_global_cd_figure.show()

assert np.isfinite(cl_global_wing_beta).all()
assert np.isfinite(cd_wing_beta).all()

beta=-12°: CL=0.310160, induced CD=0.002469, panels=180
beta=-6°: CL=0.315352, induced CD=0.002233, panels=180
beta=+0°: CL=0.317089, induced CD=0.002153, panels=180
beta=+6°: CL=0.315352, induced CD=0.002233, panels=180
beta=+12°: CL=0.310160, induced CD=0.002469, panels=180
